# Task 48: Manual Exhaustive Hyperparameter Search

Enumerate every learning-rate, regularization, and iteration combination, score each across stratified folds, and report final performance without using a built-in grid-search tool.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates=[Path('heart.csv'),Path('Task_48_Manual_Exhaustive_Hyperparameter_Search/heart.csv'),Path('Heart_Disease_Tasks/Task_48_Manual_Exhaustive_Hyperparameter_Search/heart.csv')]
data=pd.read_csv(next(path for path in candidates if path.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[features].to_numpy(float),data.target.to_numpy(int)

def split(labels,fraction,seed):
    rng=np.random.default_rng(seed);a=[];b=[]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);rng.shuffle(ids);cut=round(fraction*len(ids));a.extend(ids[:cut]);b.extend(ids[cut:])
    rng.shuffle(a);rng.shuffle(b);return np.array(a),np.array(b)

trainval,test=split(y,.8,48);Xdev,Xtest=X[trainval],X[test];ydev,ytest=y[trainval],y[test]
def folds_for(labels,k=4,seed=48):
    rng=np.random.default_rng(seed);folds=[[] for _ in range(k)]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);rng.shuffle(ids)
        for index,part in enumerate(np.array_split(ids,k)):folds[index].extend(part.tolist())
    return [np.array(sorted(part)) for part in folds]

folds=folds_for(ydev)
class ManualLogistic:
    def __init__(self,rate,steps,l2):self.rate,self.steps,self.l2=rate,steps,l2
    def fit(self,x,y):
        self.w=np.zeros(x.shape[1]);self.b=0.
        for _ in range(self.steps):
            p=1/(1+np.exp(-np.clip(x@self.w+self.b,-35,35)));e=p-y
            self.w-=self.rate*(x.T@e/len(y)+self.l2*self.w);self.b-=self.rate*e.mean()
        return self
    def predict(self,x):return (1/(1+np.exp(-np.clip(x@self.w+self.b,-35,35)))>=.5).astype(int)

def f1(actual,pred):
    tp=np.sum((actual==1)&(pred==1));fp=np.sum((actual==0)&(pred==1));fn=np.sum((actual==1)&(pred==0))
    precision=tp/(tp+fp) if tp+fp else 0.;recall=tp/(tp+fn) if tp+fn else 0.
    return 2*precision*recall/(precision+recall) if precision+recall else 0.

# Explicit nested loops exhaustively visit every parameter tuple and fold.
regularization_values=[.001,.01,.05]
learning_rate_values=[.03,.06,.1]
iteration_values=[700,1200]
results=[]
for l2 in regularization_values:
    for rate in learning_rate_values:
        for steps in iteration_values:
            fold_scores=[]
            for valid in folds:
                valid_set=set(valid.tolist());train=np.array([i for i in range(len(ydev)) if i not in valid_set])
                xtr,xv=Xdev[train],Xdev[valid];ytr,yv=ydev[train],ydev[valid]
                mean,scale=xtr.mean(0),xtr.std(0);scale[scale==0]=1
                xtr,xv=(xtr-mean)/scale,(xv-mean)/scale
                model=ManualLogistic(rate,steps,l2).fit(xtr,ytr)
                fold_scores.append(f1(yv,model.predict(xv)))
            results.append({'l2':l2,'learning_rate':rate,'iterations':steps,'mean_f1':float(np.mean(fold_scores)),'std_f1':float(np.std(fold_scores,ddof=1))})

ranking=pd.DataFrame(results).sort_values(['mean_f1','std_f1'],ascending=[False,True]).reset_index(drop=True)
best=ranking.iloc[0]
mean,scale=Xdev.mean(0),Xdev.std(0);scale[scale==0]=1
xdev,xtest=(Xdev-mean)/scale,(Xtest-mean)/scale
final_model=ManualLogistic(float(best.learning_rate),int(best.iterations),float(best.l2)).fit(xdev,ydev)
pred=final_model.predict(xtest)
tp=np.sum((ytest==1)&(pred==1));tn=np.sum((ytest==0)&(pred==0));fp=np.sum((ytest==0)&(pred==1));fn=np.sum((ytest==1)&(pred==0))
precision=tp/(tp+fp) if tp+fp else 0.;recall=tp/(tp+fn) if tp+fn else 0.
print(f'Exhaustively evaluated {len(results)} configurations on {len(folds)} folds')
display(ranking.head(8).round(4))
print('Selected settings:',{'l2':float(best.l2),'learning_rate':float(best.learning_rate),'iterations':int(best.iterations)})
print('Held-out test metrics:',{'accuracy':float((tp+tn)/len(ytest)),'precision':float(precision),'recall':float(recall),'f1':float(2*precision*recall/(precision+recall)) if precision+recall else 0.})
print('The test split was held out from grid selection.')

Exhaustively evaluated 18 configurations on 4 folds


,l2,learning_rate,iterations,mean_f1,std_f1
0,0.001,0.03,700,0.8684,0.0340
1,0.010,0.03,700,0.8684,0.0340
2,0.050,0.06,700,0.8684,0.0340
3,0.050,0.06,1200,0.8684,0.0340
4,0.050,0.10,700,0.8684,0.0340
5,0.050,0.10,1200,0.8684,0.0340
6,0.050,0.03,1200,0.8655,0.0387
7,0.050,0.03,700,0.8624,0.0405


Selected settings: {'l2': 0.001, 'learning_rate': 0.03, 'iterations': 700}
Held-out test metrics: {'accuracy': 0.8360655737704918, 'precision': 0.8285714285714286, 'recall': 0.8787878787878788, 'f1': 0.8529411764705883}
The test split was held out from grid selection.
